# IMDb — Films Set in Turin Scraper

IMDb allows filtering films by filming location. I used this to build a dataset of films shot in Turin, collecting title, year, genres, director, poster, and production stills for each one.

The search results page uses infinite scroll — clicking a "50 more" button loads the next batch of results without changing the URL. Standard pagination scraping doesn't work here. Instead, Selenium repeatedly clicks that button until all results are loaded, then visits each film page individually to extract metadata and images.

The scraper is split into four stages so each can be run independently:
1. Scrape all film metadata
2. Download posters into `posters/`
3. Download full photo collections into `collection/`
4. Save everything to `films.csv`

All outputs are saved in the same folder as this notebook.

## 1. Setup

All paths are defined relative to a hardcoded `HERE` variable pointing to the notebook's folder. Output directories are created automatically. I use `webdriver-manager` to handle ChromeDriver versioning automatically.

In [1]:
!pip install selenium webdriver-manager pandas requests -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
import os, re, time
import requests
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Optional
from collections import Counter
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

HERE          = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB"

START_URL     = "https://www.imdb.com/search/title/?locations=Torino%2C+Italy"
CSV_PATH      = os.path.join(HERE, "films.csv")
POSTERS_DIR   = os.path.join(HERE, "posters")
COLLECTION_DIR= os.path.join(HERE, "collection")
MAX_LOAD_MORE = 30
MAX_MOVIES    = 500
HEADLESS      = False

os.makedirs(POSTERS_DIR,    exist_ok=True)
os.makedirs(COLLECTION_DIR, exist_ok=True)

print("CSV path    :", CSV_PATH)
print("Posters     :", POSTERS_DIR)
print("Collection  :", COLLECTION_DIR)

CSV path    : C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\films.csv
Posters     : C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\posters
Collection  : C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\collection


## 2. Browser and helper functions

The `make_driver` function creates a configured Chrome instance. `accept_cookies` dismisses the IMDb cookie banner if it appears. `safe_filename` sanitises film titles for use as filenames. `fast_download` uses a persistent `requests.Session` which reuses HTTP connections across all downloads — significantly faster than opening a new connection per image.

In [3]:
def make_driver(headless: bool = False) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1600,2000")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.set_page_load_timeout(40)
    return driver


def accept_cookies(driver: webdriver.Chrome):
    """Dismiss the IMDb cookie banner if it appears."""
    try:
        btn = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//button[contains(., 'Accept') or @id='onetrust-accept-btn-handler']"
            ))
        )
        btn.click()
        time.sleep(0.5)
    except Exception:
        pass


def safe_filename(text: str) -> str:
    return re.sub(r'[\\/:*?"<>|]', "", text).replace(" ", "_")[:80]


# Persistent session for faster downloads
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

def fast_download(url: str, filepath: str) -> bool:
    """Download an image, skipping if it already exists."""
    if not url or os.path.exists(filepath):
        return True
    try:
        r = session.get(url, timeout=10, stream=True)
        if r.status_code == 200:
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(8192):
                    if chunk:
                        f.write(chunk)
            return True
    except Exception:
        pass
    return False

## 3. Loading all search results

IMDb loads 50 results at a time behind a "50 more" button. I scroll to the bottom of the page and click it repeatedly until no more results load, expanding the entire result set onto a single page before collecting URLs.

In [4]:
def expand_results(driver: webdriver.Chrome, max_clicks: int = MAX_LOAD_MORE):
    clicks = 0
    while clicks < max_clicks:
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.0)
            button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[.//span[contains(@class,'ipc-see-more__text') "
                    "and contains(., '50 more')]]"
                ))
            )
            driver.execute_script("arguments[0].click();", button)
            time.sleep(2.0)
            clicks += 1
            print(f"  Loaded more results (click {clicks})")
        except Exception:
            print(f"  No more results to load. Total clicks: {clicks}")
            break


def collect_film_urls(driver: webdriver.Chrome) -> List[str]:
    """Collect all unique film URLs from the fully expanded results page."""
    urls  = set()
    cards = driver.find_elements(
        By.CSS_SELECTOR, "div.sc-fc35a1ef-1.lmHCrT.dli-parent"
    )
    print(f"Found {len(cards)} result cards")
    for card in cards:
        try:
            a    = card.find_element(By.CSS_SELECTOR, "a.ipc-title-link-wrapper")
            href = a.get_attribute("href")
            if href and "/title/tt" in href:
                urls.add(href.split("?")[0])
        except Exception:
            pass
    result = sorted(urls)
    print(f"Unique film URLs: {len(result)}")
    return result

## 4. Scraping individual film pages

For each film I extract title, release year, genres, director, poster image URL, and up to 6 photo URLs from the Photos section. The `MovieRecord` dataclass keeps each film's data structured and consistent throughout the pipeline.

In [5]:
@dataclass
class MovieRecord:
    url:         str
    title:       str
    year:        str
    genres:      str
    director:    str
    poster_file: Optional[str] = None
    photo_files: List[str]     = field(default_factory=list)


def scrape_film_page(driver: webdriver.Chrome, url: str):
    driver.get(url)
    accept_cookies(driver)
    time.sleep(0.8)

    title, year, genres, director = "", "", "", ""
    poster_url  = None
    photos_urls = []

    try:
        title = driver.find_element(
            By.CSS_SELECTOR, "[data-testid='hero-title-block__title']"
        ).text.strip()
    except Exception:
        try:
            title = driver.find_element(By.CSS_SELECTOR, "h1").text.strip()
        except Exception:
            pass

    try:
        year_el = driver.find_element(By.XPATH, "//a[contains(@href,'releaseinfo')]")
        matches = re.findall(r"(19\d{2}|20\d{2})", year_el.text)
        if matches:
            year = matches[0]
    except Exception:
        pass

    try:
        genre_els = driver.find_elements(
            By.XPATH, "//a[contains(@href,'/search/title/?genres=')]"
        )
        genres = ", ".join(dict.fromkeys(
            g.text.strip() for g in genre_els if g.text.strip()
        ))
    except Exception:
        pass

    try:
        name_links = driver.find_elements(
            By.XPATH, "//a[contains(@href, '/name/nm')]"
        )
        if name_links:
            director = name_links[0].text.strip()
    except Exception:
        pass

    try:
        poster_el  = driver.find_element(
            By.CSS_SELECTOR, "[data-testid='hero-media__poster'] img.ipc-image"
        )
        poster_url = poster_el.get_attribute("src") or ""
        if " " in poster_url:
            poster_url = poster_url.split(",")[0].split(" ")[0]
    except Exception:
        pass

    try:
        photo_grid = driver.find_elements(
            By.CSS_SELECTOR, "[data-testid='photos-image-grid']"
        )
        imgs = []
        if photo_grid:
            imgs = photo_grid[0].find_elements(By.CSS_SELECTOR, "img.ipc-image")
        else:
            for h3 in driver.find_elements(By.XPATH, "//h3[contains(., 'Photos')]"):
                section = h3.find_element(By.XPATH, "./ancestor::section[1]")
                imgs    = section.find_elements(By.CSS_SELECTOR, "img.ipc-image")
                if imgs:
                    break
        for img in imgs:
            if len(photos_urls) >= 6:
                break
            src = img.get_attribute("src") or ""
            if " " in src:
                src = src.split(",")[0].split(" ")[0]
            if src and src not in photos_urls and src != poster_url:
                photos_urls.append(src)
    except Exception:
        pass

    return MovieRecord(
        url=url, title=title, year=year, genres=genres, director=director
    ), poster_url, photos_urls

## 5. Scraping all films

This cell opens the search page, expands all results, then visits each film page to collect metadata and image URLs. The browser closes as soon as all pages are scraped — image downloads happen separately in the next cells using a faster requests-based approach.

In [6]:
def run_scraper():
    driver  = make_driver(HEADLESS)
    records = []

    try:
        print(f"Opening: {START_URL}")
        driver.get(START_URL)
        accept_cookies(driver)
        time.sleep(1.0)

        expand_results(driver, MAX_LOAD_MORE)
        urls = collect_film_urls(driver)

        if not urls:
            print("No films found — check the CSS selectors.")
            return []

        if len(urls) > MAX_MOVIES:
            urls = urls[:MAX_MOVIES]
            print(f"Capped at {MAX_MOVIES} films.")

        for idx, url in enumerate(urls, 1):
            print(f"[{idx}/{len(urls)}] {url}")
            record, poster_url, photo_urls = scrape_film_page(driver, url)
            records.append((record, poster_url, photo_urls))
            print(f"  {record.title} ({record.year}) — {len(photo_urls)} photos")

    finally:
        driver.quit()

    print(f"\nScraping done. {len(records)} films collected.")
    return records


scraped = run_scraper()

Opening: https://www.imdb.com/search/title/?locations=Torino%2C+Italy
  Loaded more results (click 1)
  Loaded more results (click 2)
  Loaded more results (click 3)
  Loaded more results (click 4)
  Loaded more results (click 5)
  No more results to load. Total clicks: 5
Found 300 result cards
Unique film URLs: 286
[1/286] https://www.imdb.com/title/tt0045286/
  La pattuglia sperduta (1954) — 0 photos
[2/286] https://www.imdb.com/title/tt0062707/
  The Violent Four (1968) — 6 photos
[3/286] https://www.imdb.com/title/tt0065577/
  The Couples (1970) — 4 photos
[4/286] https://www.imdb.com/title/tt0065761/
  The Cat o' Nine Tails (1971) — 6 photos
[5/286] https://www.imdb.com/title/tt0067178/
  They Have Changed Their Face (1971) — 6 photos
[6/286] https://www.imdb.com/title/tt0073734/
  Il sospetto (1975) — 5 photos
[7/286] https://www.imdb.com/title/tt0078408/
  Double Game (1977) — 6 photos
[8/286] https://www.imdb.com/title/tt0082278/
  Don't Look in the Attic (1982) — 6 photos
[9/2

## 6. Downloading posters

Poster images are saved to the `posters/` folder. Each file is named after the film title and year. Files that already exist are skipped automatically so this cell can be re-run safely.

In [7]:
print(f"Downloading posters for {len(scraped)} films...")
ok, fail = 0, 0

for record, poster_url, photo_urls in scraped:
    base = safe_filename(f"{record.title or 'film'}_{record.year or 'NA'}")
    if poster_url:
        pfile = os.path.join(POSTERS_DIR, f"{base}_poster.jpg")
        if fast_download(poster_url, pfile):
            record.poster_file = os.path.basename(pfile)
            ok += 1
            print(f"  {record.title}")
        else:
            fail += 1

print(f"\nPosters done. Saved: {ok} | Failed: {fail}")
print(f"Folder: {POSTERS_DIR}")

  La pattuglia sperduta
  The Violent Four
  The Couples
  The Cat o' Nine Tails
  They Have Changed Their Face
  Il sospetto
  Double Game
  Don't Look in the Attic
  Girls Will Be Girls
  I due carabinieri
  Razza selvaggia
  The Young Nun
  The Amazing Race
  Elisa di Rivombrosa
  Do You Like Hitchcock?
  Mana: Beyond Belief
  Hans
  Pianosequenza
  Our City Dreams
  Citizen Sam
  Einstein
  Confessione
  Conseguenze
  Margherita Vicario: Mandela
  Franco e Gianni - Una storia di Torino
  Made in Italy
  Pastrone!
  Un caso di coscienza
  Vado a messa
  Wild Blood
  Extravergine
  I'm in Love with My Car
  Interplay
  Songs with other strangers
  Il vaso
  Ragazzi irresponsabili
  L'atlante della memoria
  Fame
  L'eco del passato
  Zheng
  Dear Dad
  Un giorno qualsiasi
  Fuori fuoco
  La stanza
  Desensibilizzazione ambientale
  La vita di un amore
  I 30 anni
  Io sono un robot
  Fino all'ultimo respiro
  Corto lasciato a metà
  The Man Who Loves
  Good and Evil
  Stealing Realit

## 7. Downloading photo collections

Up to 6 production stills per film are saved to the `collection/` folder. These come from the Photos section on each film's IMDb page. Files are named by film title and photo index.

In [8]:
print(f"Downloading photo collections for {len(scraped)} films...")
ok, fail = 0, 0

for record, poster_url, photo_urls in scraped:
    base = safe_filename(f"{record.title or 'film'}_{record.year or 'NA'}")
    for i, purl in enumerate(photo_urls, 1):
        pfile = os.path.join(COLLECTION_DIR, f"{base}_photo{i}.jpg")
        if fast_download(purl, pfile):
            record.photo_files.append(os.path.basename(pfile))
            ok += 1
        else:
            fail += 1

print(f"\nCollection done. Saved: {ok} | Failed: {fail}")
print(f"Folder: {COLLECTION_DIR}")


Collection done. Saved: 665 | Failed: 0
Folder: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\collection


## 8. Saving the CSV

All film metadata is saved to `films.csv`. Each row contains the film URL, title, year, genres, director, poster filename, and a semicolon-separated list of photo filenames.

In [9]:
rows = [{
    "url":         r.url,
    "title":       r.title,
    "year":        r.year,
    "genres":      r.genres,
    "director":    r.director,
    "poster_file": r.poster_file or "",
    "photo_files": ";".join(r.photo_files)
} for r, _, _ in scraped]

df_films = pd.DataFrame(rows)
df_films.to_csv(CSV_PATH, index=False)
print(f"CSV saved: {CSV_PATH}")
print(f"Total films: {len(df_films)}")
df_films.head(10)

CSV saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\films.csv
Total films: 286


,url,title,year,genres,director,poster_file,photo_files
0,https://www.imdb.com/title/tt0045286/,La pattuglia sperduta,1954,,Piero Nelli,La_pattuglia_sperduta_1954_poster.jpg,
1,https://www.imdb.com/title/tt0062707/,The Violent Four,1968,,Carlo Lizzani,The_Violent_Four_1968_poster.jpg,The_Violent_Four_1968_photo1.jpg;The_Violent_F...
2,https://www.imdb.com/title/tt0065577/,The Couples,1970,,Vittorio De Sica,The_Couples_1970_poster.jpg,The_Couples_1970_photo1.jpg;The_Couples_1970_p...
3,https://www.imdb.com/title/tt0065761/,The Cat o' Nine Tails,1971,,Dario Argento,The_Cat_o'_Nine_Tails_1971_poster.jpg,The_Cat_o'_Nine_Tails_1971_photo1.jpg;The_Cat_...
4,https://www.imdb.com/title/tt0067178/,They Have Changed Their Face,1971,,Corrado Farina,They_Have_Changed_Their_Face_1971_poster.jpg,They_Have_Changed_Their_Face_1971_photo1.jpg;T...
5,https://www.imdb.com/title/tt0073734/,Il sospetto,1975,,Francesco Maselli,Il_sospetto_1975_poster.jpg,Il_sospetto_1975_photo1.jpg;Il_sospetto_1975_p...
6,https://www.imdb.com/title/tt0078408/,Double Game,1977,,Carlo Ausino,Double_Game_1977_poster.jpg,Double_Game_1977_photo1.jpg;Double_Game_1977_p...
7,https://www.imdb.com/title/tt0082278/,Don't Look in the Attic,1982,,Carlo Ausino,Don't_Look_in_the_Attic_1982_poster.jpg,Don't_Look_in_the_Attic_1982_photo1.jpg;Don't_...
8,https://www.imdb.com/title/tt0082455/,Girls Will Be Girls,1980,,Mariano Laurenti,Girls_Will_Be_Girls_1980_poster.jpg,Girls_Will_Be_Girls_1980_photo1.jpg;Girls_Will...
9,https://www.imdb.com/title/tt0087181/,I due carabinieri,1984,,Carlo Verdone,I_due_carabinieri_1984_poster.jpg,I_due_carabinieri_1984_photo1.jpg;I_due_carabi...


In [5]:
#added this because i saw that i had no genres in the csv
import re, time, requests, pandas as pd
from collections import Counter

CSV_PATH = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\films.csv"

def get_genres_from_imdb(url: str) -> str:
    m = re.search(r'tt(\d+)', str(url))
    if not m:
        return ''
    imdb_url = f"https://www.imdb.com/title/tt{m.group(1)}/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9"
    }
    try:
        r = requests.get(imdb_url, headers=headers, timeout=15)
        if r.status_code != 200:
            return ''
        matches = re.findall(r'"genre"\s*:\s*\[([^\]]+)\]', r.text)
        if not matches:
            matches = re.findall(r'"genre"\s*:\s*"([^"]+)"', r.text)
            return matches[0] if matches else ''
        genres = re.findall(r'"([^"]+)"', matches[0])
        return ', '.join(genres)
    except Exception as e:
        print(f"  Error for {url}: {e}")
        return ''

df_films = pd.read_csv(CSV_PATH)
print(f"Enriching {len(df_films)} films...")

genres_list = []
for i, row in df_films.iterrows():
    g = get_genres_from_imdb(row['url'])
    genres_list.append(g)
    if g:
        print(f"  [{i+1}] {row['title']} → {g}")
    else:
        print(f"  [{i+1}] {row['title']} → (none found)")
    time.sleep(0.8)

df_films['genres'] = genres_list
df_films.to_csv(CSV_PATH, index=False)

print(f"\nDone.")
print(f"Films with genres: {(df_films['genres'] != '').sum()} / {len(df_films)}")
print("\nTop genres:")
all_genres = [g.strip() for genres in df_films['genres'].dropna()
              for g in str(genres).split(',') if g.strip()]
print(Counter(all_genres).most_common(10))

Enriching 286 films...
  [1] La pattuglia sperduta → (none found)
  [2] The Violent Four → Crime, Drama
  [3] The Couples → Comedy, Drama
  [4] The Cat o' Nine Tails → Horror, Mystery, Thriller
  [5] They Have Changed Their Face → Drama, Horror
  [6] Il sospetto → Crime, Drama
  [7] Double Game → Crime, Drama, Thriller
  [8] Don't Look in the Attic → Horror, Mystery
  [9] Girls Will Be Girls → Comedy
  [10] I due carabinieri → Comedy, Crime
  [11] Il compagno → Drama, Romance
  [12] Razza selvaggia → Drama
  [13] The Young Nun → Drama, Romance
  [14] Aldo dice 26x1 → (none found)
  [15] The Amazing Race → Adventure, Family, Game-Show
  [16] Elisa di Rivombrosa → Drama, History, Romance
  [17] Do You Like Hitchcock? → Mystery, Thriller
  [18] Mana: Beyond Belief → Documentary
  [19] Hans → Drama
  [20] Pianosequenza → Drama
  [21] Our City Dreams → Documentary
  [22] La maschera d'acqua → Comedy, Drama, Romance
  [23] Citizen Sam → Documentary
  [24] Einstein → Biography, Drama
  [25] C

In [4]:
df_films.to_csv(CSV_PATH, index=False)
print("Saved.")
print(f"Films with genres: {(df_films['genres'] != '').sum()} / {len(df_films)}")

Saved.
Films with genres: 284 / 286


In [4]:
import requests, time, re, pandas as pd

CSV_PATH = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\films.csv"

def get_plot_summary(url):
    m = re.search(r'tt(\d+)', str(url))
    if not m:
        return ''
    try:
        r = requests.get(
            f"https://www.imdb.com/title/tt{m.group(1)}/plotsummary/",
            headers={
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                              "AppleWebKit/537.36 (KHTML, like Gecko) "
                              "Chrome/120.0.0.0 Safari/537.36",
                "Accept-Language": "en-US,en;q=0.9",
                "Accept": "text/html,application/xhtml+xml"
            },
            timeout=15
        )
        patterns = [
            r'"plot":\{"__typename":"Plot","plotText":\{"__typename":"Markdown","plainText":"([^"]{20,600})"',
            r'class="ipc-html-content-inner-div"[^>]*>([^<]{20,500})<',
            r'"summaries":\[.*?"text":"([^"]{20,600})"',
            r'data-testid="plot-summary-text"[^>]*>([^<]{20,500})<',
        ]
        for pat in patterns:
            matches = re.findall(pat, r.text)
            if matches:
                return matches[0].strip()
        return ''
    except:
        return ''

df_films = pd.read_csv(CSV_PATH)
print(f"Fetching plot summaries for {len(df_films)} films...")

plots = []
for i, row in df_films.iterrows():
    p = get_plot_summary(row['url'])
    plots.append(p)
    if p:
        print(f"  [{i+1}] {row['title']} → {p[:80]}...")
    else:
        print(f"  [{i+1}] {row['title']} → (none)")
    time.sleep(0.8)

df_films['plot_summary'] = plots
df_films.to_csv(CSV_PATH, index=False)
print(f"\nDone. {sum(1 for p in plots if p)} / {len(plots)} summaries added.")

Fetching plot summaries for 286 films...
  [1] La pattuglia sperduta → The adventures of Lieutenant Airoldi of Piedmont who is trying to gather intelli...
  [2] The Violent Four → A detective is assigned to head a manhunt for four violent bank robbers....
  [3] The Couples → Three ordinary couples deal with surreal situations: buying a refrigerator, cele...
  [4] The Cat o' Nine Tails → A reporter and a blind puzzle writer investigate a series of murders connected t...
  [5] They Have Changed Their Face → In this allegory on capitalism, director of a known car corporation invites one ...
  [6] Il sospetto → Fascism has forced the leadership of the Italian Communist Party to settle in Pa...
  [7] Double Game → Drugs, murders, robberies. In Turin, violence is an everyday fact of life, but a...
  [8] Don't Look in the Attic → A couple inherits a mansion, but when they move in they discover that it is haun...
  [9] Girls Will Be Girls → A group of employees, on a winter vacation, rather th

## 9. Dataset overview

In [8]:
from collections import Counter

df = pd.read_csv(CSV_PATH)
print(f"Total films  : {len(df)}")
print(f"Columns      : {list(df.columns)}")
print(f"Films with plot summary: {df['plot_summary'].notna().sum()} / {len(df)}")
print()
print("Top genres:")
all_genres = [
    g.strip()
    for genres in df["genres"].dropna()
    for g in genres.split(",")
    if g.strip()
]
print(Counter(all_genres).most_common(10))
print()
print("Films per decade:")
df["decade"] = (pd.to_numeric(df["year"], errors="coerce") // 10 * 10).astype("Int64")
print(df["decade"].value_counts().sort_index())
print()
df[["title", "year", "genres", "director"]].head(10)

Total films  : 286
Columns      : ['url', 'title', 'year', 'genres', 'director', 'poster_file', 'photo_files', 'plot_summary']
Films with plot summary: 145 / 286

Top genres:
[('Short', 119), ('Drama', 86), ('Documentary', 68), ('Comedy', 45), ('Crime', 18), ('Thriller', 18), ('Music', 16), ('Biography', 13), ('Horror', 10), ('Romance', 10)]

Films per decade:
decade
1900      2
1940      1
1950      2
1960      3
1970      5
1980      6
1990      2
2000     49
2010    118
2020     87
Name: count, dtype: Int64



,title,year,genres,director
0,La pattuglia sperduta,1954.0,NaN,Piero Nelli
1,The Violent Four,1968.0,"Crime, Drama",Carlo Lizzani
2,The Couples,1970.0,"Comedy, Drama",Vittorio De Sica
3,The Cat o' Nine Tails,1971.0,"Horror, Mystery, Thriller",Dario Argento
4,They Have Changed Their Face,1971.0,"Drama, Horror",Corrado Farina
5,Il sospetto,1975.0,"Crime, Drama",Francesco Maselli
6,Double Game,1977.0,"Crime, Drama, Thriller",Carlo Ausino
7,Don't Look in the Attic,1982.0,"Horror, Mystery",Carlo Ausino
8,Girls Will Be Girls,1980.0,Comedy,Mariano Laurenti
9,I due carabinieri,1984.0,"Comedy, Crime",Carlo Verdone
